# Prepare regions for upload to MorphoSource

We looked into uploading datasets for sharing, see https://github.com/habi/sticklebacks-manuscript/issues/11
We figured out, that [MorphoSource](https://www.morphosource.org/) is probably the best thing to try.
This notebook is used to prepare the `rec_regions` exports written by the `BucketSeparator.ipynb` notebook for upload to there.

The cells below are used to set up the whole notebook.
They load needed libraries and set some default values.

In [ ]:
# Load the modules we need
import platform
import os
import glob
import pandas
import imageio
import numpy
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
import seaborn
import dask
import dask_image.imread
from dask.distributed import Client, LocalCluster
from tqdm.auto import tqdm, trange

In [ ]:
# Load our own log file parsing code
# This is loaded as a submodule to alleviate excessive copy-pasting between *all* projects we do
# See https://github.com/habi/BrukerSkyScanLogfileRuminator for details on its inner workings
import BrukerSkyScanLogfileRuminator.parsing_functions as logparse

In [ ]:
# Set dask temporary folder
# Do this before creating a client: https://stackoverflow.com/a/62804525/323100
# We use the fast internal SSD for speed reasons
import tempfile
if 'Linux' in platform.system():
    # Check if me mounted the FastSSD, otherwise go to standard tmp file
    if os.path.exists(os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')):
        tmp = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD', 'tmp')
    else:
        tmp = tempfile.gettempdir()
elif 'Darwin' in platform.system():
    tmp = tempfile.gettempdir()
else:
    if 'anaklin' in platform.node():
        tmp = os.path.join('F:\\tmp')
    else:
        tmp = os.path.join('D:\\tmp')
dask.config.set({'temporary_directory': tmp})
print('Dask temporary files go to %s' % dask.config.get('temporary_directory'))

In [ ]:
from dask.distributed import Client
client = Client()

In [ ]:
# Set up figure defaults
plt.rc('image', cmap='gray', interpolation='nearest')  # Display all images in b&w and with 'nearest' interpolation
# plt.rcParams['figure.figsize'] = (16 * 0.618, 9 * 0.618)  # Size up figures a bit
plt.rcParams['figure.dpi'] = 300

In [ ]:
# Setup scale bar defaults
plt.rcParams['scalebar.location'] = 'lower right'
plt.rcParams['scalebar.frameon'] = False
plt.rcParams['scalebar.color'] = 'white'

In [ ]:
# Display all plots identically
lines = 3
# And then do something like
# plt.subplot(lines, int(numpy.ceil(len(Data) / float(lines))), c + 1)

Since the (tomographic) data can reside on different drives we set a folder to use below

In [ ]:
# Different locations if running either on Linux or Windows
FastSSD = True
if 'Linux' in platform.system():
    if FastSSD:
        BasePath = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')
    else:
        BasePath = os.path.join(os.path.sep, 'home', 'habi', 'research_storage_ben', 'microCT_Stickleback')
elif 'Windows' in platform.system():
    if FastSSD:
        BasePath = os.path.join('F:\\')
    else:
        BasePath = os.path.join('N:\\')
if 'research_storage_ben' in BasePath:
    Root = os.path.join(BasePath)
else:
    Root = os.path.join(BasePath, 'IEE Stickleback')
# Force reading from Bens research storage folder
# Root = os.path.join(os.path.sep, 'home', 'habi', 'research_storage_ben', 'microCT_Stickleback')
print('We are loading all the data from %s' % Root)

Now that we are set up, actually start to load/ingest the data.

In [ ]:
# Make us a dataframe for saving all that we need
Data = pandas.DataFrame()

In [ ]:
# Get *all* log files present on disk
# Using os.walk is way faster than using recursive glob.glob
# Not sorting the found logfiles is also making it quicker
Data['LogFile'] = [os.path.join(root, name)
                   for root, dirs, files in os.walk(Root)
                   for name in files
                   if name.endswith((".log"))]

In [ ]:
# Get all folders
Data['Folder'] = [os.path.dirname(f) for f in Data['LogFile']]

In [ ]:
# Show a (small) sampler of the loaded data as a first check
Data.sample(n=5)

In [ ]:
# Check for samples which are not yet reconstructed
for c, row in Data.iterrows():
    # Iterate over every 'proj' folder
    if 'proj' in row.Folder:
        if 'TScopy' not in row.Folder and 'PR' not in row.Folder:
            # If there's nothing with 'rec*' on the same level, then tell us
            if not glob.glob(row.Folder.replace('proj', '*rec*')):
                print('- %s is missing matching reconstructions' % row.LogFile[len(Root) + 1:])
# Sticklebucket_14/proj2/Sticklebucket_14~00.log and 
# Sticklebucket_15/proj2/Sticklebucket_15~00.log are failed scans where we cannot do a reconstruction

In [ ]:
# Search for any .csv files in each folder.
# These are only generated when the "X/Y Alignment With a Reference Scan" was performed in NRecon.
# If those files do *not* exist we have missed to do it and should correct for this.
Data['XYAlignment'] = [glob.glob(os.path.join(f, '*T*.csv')) for f in Data['Folder']]

In [ ]:
# Display samples which are missing the .csv-files for the XY-alignment
for c, row in Data.iterrows():
    # Iterate over every 'proj' folder
    if 'proj' in row['Folder']:
        if not row['XYAlignment']:
            if not any(x in row.LogFile for x in ['rectmp.log',  # because we only exclude temporary logfiles in a later step
                                                  'proj_nofilter',  # since these two scans of single teeth don't contain a reference scan
                                                  'TScopy',  # discard *t*hermal *s*hift data
                                                  ]):
                print('- %s has *not* been X/Y aligned' % row.LogFile[len(Root) + 1:])

In [ ]:
# Get rid of all the logfiles from all the folders that might be on disk but that we don't want to load the data from
for c, row in Data.iterrows():
    if os.path.split(row.Folder)[-1] == 'proj':  # drop all projections folders
        Data.drop([c], inplace=True)
    elif 'ucket' not in row.Folder:  # Remove all test scans which are not named 'Sticklbucket_*' or something else containing 'ucket'
        Data.drop([c], inplace=True)
    elif '_regions' in row.Folder:  # Exclude all log files that we write in ourselves
        Data.drop([c], inplace=True)
    elif os.path.split(row.LogFile)[1].startswith('._'):  # Remove macos metadata files for files on external storage
        Data.drop([c], inplace=True)        
# Reset dataframe to something that we would get if we only would have loaded the 'rec' files
Data = Data.reset_index(drop=True)

It's a bit silly to exclude the self-written log files (`*_regions/*/*.log`) above, but like so we know for sure to include everything we've exported...

In [ ]:
# Generate us some meaningful colums in the dataframe
Data['Sample'] = [os.path.basename(log).replace('_rec.log', '') for log in Data['LogFile']]
Data['Scan'] = [os.path.basename(os.path.dirname(log)) for log in Data['LogFile']]

In [ ]:
# Show the data from the last loaded scans
Data.tail(n=5)

In [ ]:
# Load the file names of all the reconstructions of all the scans
Data['Filenames Reconstructions'] = [sorted(glob.glob(os.path.join(f, '*rec0*.png'))) for f in Data['Folder']]
# How many reconstructions do we have?
Data['Number of reconstructions'] = [len(r) for r in Data['Filenames Reconstructions']]

In [ ]:
# Drop samples which have either not been reconstructed yet or of which we deleted the reconstructions with
# `find . -name "*rec*.png" -type f -mtime +333 -delete`
# Based on https://stackoverflow.com/a/13851602
# for c,row in Data.iterrows():
#     if not row['Number of reconstructions']:
#         print('%s contains no PNG files, we might be currently reconstructing it' % row.Folder)
Data = Data[Data['Number of reconstructions'] > 0]
# Reset the dataframe count/index for easier indexing afterwards
Data.reset_index(drop=True, inplace=True)
print('We have %s folders with reconstructions' % (len(Data)))

In [ ]:
# Get parameters to doublecheck from logfiles
Data['Voxelsize'] = [logparse.pixelsize(log) for log in Data['LogFile']]
Data['Filter'] = [logparse.whichfilter(log) for log in Data['LogFile']]
Data['Exposuretime'] = [logparse.exposuretime(log) for log in Data['LogFile']]
Data['Scanner'] = [logparse.scanner(log) for log in Data['LogFile']]
Data['Averaging'] = [logparse.averaging(log) for log in Data['LogFile']]
Data['ProjectionSize'] = [logparse.projection_size(log) for log in Data['LogFile']]
Data['RotationStep'] = [logparse.rotationstep(log) for log in Data['LogFile']]
Data['Grayvalue'] = [logparse.reconstruction_grayvalue(log) for log in Data['LogFile']]
Data['RingartefactCorrection'] = [logparse.ringremoval(log) for log in Data['LogFile']]
Data['BeamHardeningCorrection'] = [logparse.beamhardening(log) for log in Data['LogFile']]
Data['DefectPixelMasking'] = [logparse.defectpixelmasking(log) for log in Data['LogFile']]
Data['Scan date'] = [logparse.scandate(log) for log in Data['LogFile']]

In [ ]:
# Sort dataframe based on the scan date
Data.sort_values(by=['Scan date'],
                 ignore_index=True,
                 inplace=True)

Now we 'load' all reconstructions from disks into stacks.

In [ ]:
# Load all reconstructions into ephemereal DASK arrays, with a nice progress bar...
Reconstructions = [None] * len(Data)
for c, row in tqdm(Data.iterrows(),
                   desc='Loading reconstructions',
                   total=len(Data)):
    Reconstructions[c] = dask_image.imread.imread(os.path.join(row['Folder'], '*rec*.png'))[:, :, :, 0]  # Get rid of the color channel

In [ ]:
Reconstructions[0]

In [ ]:
# What do we have on disk?
print('We have %s reconstructions on %s' % (Data['Number of reconstructions'].sum(), Root))
print('This is about %s reconstructions per scan (%s scans in %s folders)' % (round(Data['Number of reconstructions'].sum() / len(Data)),
                                                                              len(Data),
                                                                              len(Data.Sample.unique())))

In [ ]:
# How big are the datasets?
Data['Size'] = [rec.shape for rec in Reconstructions]

In [ ]:
# The three cardinal directions
directions = ['Axial',
              'Frontal',
              'Median']

In [ ]:
# Read or calculate the directional MIPs, put them into the dataframe and save them to disk
for d, direction in enumerate(directions):
    Data['MIP_' + direction] = ''
for c, row in tqdm(Data.iterrows(), desc='Working on MIPs', total=len(Data)):
    for d, direction in tqdm(enumerate(directions),
                             desc='%s/%s' % (row['Sample'], row['Scan']),
                             leave=False,
                             total=len(directions)):
        outfilepath = os.path.join(os.path.dirname(row['Folder']),
                                   '%s.%s.MIP.%s.png' % (row['Sample'], row['Scan'], direction))
        if not os.path.exists(outfilepath):
            # Generate and save MIP
            imageio.imwrite(outfilepath, Reconstructions[c].max(axis=d).compute().astype('uint8'))
        Data.at[c, 'MIP_' + direction] = dask_image.imread.imread(outfilepath).squeeze()

Since we've done everything *correctly* in `BucketSeparator.ipynb` we can simply go through all the desired folders and pull all in from disk.
This is more efficient than re-doing the extraction from scratch :)

In [ ]:
# Construct folder name for regions folder
Data['FolderRegionsExports'] = None
for c, row in Data.iterrows():
    Data.at[c, 'FolderRegionsExports'] = os.path.join(os.path.dirname(os.path.dirname(row.LogFile)), row.Scan + '_regions')

In [ ]:
# Search for log files we've written and construct the regions names from theseFind folders in each regions folder and put them into the dataframe
Data['RegionsLogFiles'] = None
for c, row in Data.iterrows():
    Data.at[c, 'RegionsLogFiles'] = sorted(glob.glob(os.path.join(row.FolderRegionsExports, '*', '*.log')))    

In [ ]:
Data.RegionsLogFiles[3]

In [ ]:
# Construct regions name (and double-check for errors on the way)Search for log files we've written and construct the regions names from theseFind folders in each regions folder and put them into the dataframe
Data['RegionsName'] = None
Data['RegionsFolder'] = None
for c, row in Data.iterrows():
    Data.at[c, 'RegionsName'] = [os.path.splitext(os.path.basename(logfilename))[0] for logfilename in row.RegionsLogFiles]
    Data.at[c, 'RegionsFolder'] = [os.path.dirname(logfilename) for logfilename in row.RegionsLogFiles]
    for rn, rf in zip(Data.at[c, 'RegionsName'], Data.at[c, 'RegionsFolder']):
        if rn != os.path.basename(rf):  # Tested with a manual rename on disk :)
            print(f'Error: For {row.LogFile[len(Root):]}: Extracted Region name "{rn}" does not match extracted folder name "{os.path.basename(rf)}"')

In [ ]:
Data.RegionsName[3]

In [ ]:
Data.RegionsFolder[3]

In [ ]:
Data.LogFile

MorphoSource would like to ingest a ".zip containing .tif, .jpeg, .bmp, or .dcm*", see https://docs.google.com/document/d/1QByWl5t0SFD4QkdxUdoUbeTNms3HEhQYdTndo6PR-Ts/edit?tab=t.0, so we're preparing these files.
In [a test](https://www.morphosource.org/concern/media/000885110?locale=en), we've seen that a .zip with PNGs works fine, too...

Each .zip file should contain the original `proj/*.log`, `rec/*.log` and all the files from `rec_regions/FishID/*` for reproducible research.

In [ ]:
Data.RegionsFolder[3]

In [ ]:
Data.Folder[0]

In [ ]:
# Define us a "custom" zipping function
import pathlib
import zipfile

def zip_folder(folder, logfile, scan_date):
    # Generate folder names
    folder = pathlib.Path(folder)
    logfile = pathlib.Path(logfile)

    # Generate path for the zip file
    zip_path = folder.parent / (folder.name + ".zip")

    # Search for correct label-checking file
    search_string = folder.parent.name.replace("_regions", "")
    labelcheckingfile = next(
        folder.parent.parent.glob(f"*{search_string}*.Labels.Check.png"),
        None
    )

    # Create README file (with function below)
    readme_path = create_readme(folder, logfile, scan_date)

    # Actually do the zipping now
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        # Add region folder
        for file in folder.rglob("*"):
            zf.write(file, arcname=file.relative_to(folder.parent))

        # Add reconstruction log file
        zf.write(logfile, arcname=logfile.name)

        # Add label-checking file
        zf.write(labelcheckingfile, arcname=labelcheckingfile.name)

        # Add README
        zf.write(readme_path, arcname="README.md")

    # Remove temporary README
    readme_path.unlink()

    return zip_path

In [ ]:
def get_file_listing(region):
    from pathlib import Path
   
    region = Path(region)

    files = sorted([f.relative_to(region) for f in region.rglob("*") if f.is_file()])

    pngs = [f for f in files if f.suffix.lower() == ".png"]
    regionlog = [f for f in files if f.suffix.lower() == ".log"]

    return pngs, regionlog

In [ ]:
# We want to add a custom/dynamic README.md file to *every* archive, so let's generate one
from datetime import datetime

def create_readme(region, logfile, scan_date):
    region = pathlib.Path(region)
    logfile = pathlib.Path(logfile)
    pngs, regionlog = get_file_listing(region)

    if len(pngs) > 2:
        png_listing = (
            f"│   ├── {pngs[0]}\n"
            f"│   ├── {pngs[1]}\n"
            f"│   ├── ...\n"
            f"│   ├── {pngs[-1]}"
        )
    elif len(pngs) == 1:
        png_listing = "\n".join(f"│   ├── {p}" for p in pngs)
    else:
        png_listing = ""

    log_listing = "\n".join(f"│   └── {l}" for l in regionlog)

    # Search for correct label-checking file
    search_string = region.parent.name.replace("_regions", "")
    labelcheckingfile = next(
        region.parent.parent.glob(f"*{search_string}*.Labels.Check.png"),
        None
    )    

    readme = f"""# {region.name}

## Info

This archive was generated on {datetime.now().isoformat(timespec="seconds")} with [a bespoke notebook](https://github.com/habi/sticklebacks-manuscript/PrepareForMorphoSource.ipynb) for uploading extracted datasets from [our sticklebacks manuscript](https://habi.github.io/sticklebacks-manuscript/) to [MorphoSource](https://www.morphosource.org/).
It contains the {len(pngs)} cropped reconstructions for specimen **{region.name}**, which was scanned on {scan_date}, and some aditional files.

## Archive contents

```
{region.name}.zip
├── {region.name}/
{png_listing}
{log_listing}
├── {logfile.name}
├── {labelcheckingfile.name}
└── README.md
```

## Source

- `{region.name}`: Original region folder on disk from `{region.relative_to(Root)}`
- `{logfile.name}`: Log file of the reconstructions of the original multi-specimen scan copied into the archive from `{logfile.relative_to(Root)}`
- `{labelcheckingfile.name}`: Label/vial checking file generated from the the original multi-specimen scan. Generated with [the separator notebook](https://github.com/habi/sticklebacks/blob/main/BucketSeparator.ipynb) and copied into the archive from `{labelcheckingfile.relative_to(Root)}`.
- `README.md`: This file

"""

    readme_path = region.parent / "README.md"
    readme_path.write_text(readme, encoding="utf-8")

    return readme_path

In [ ]:
for c, row in tqdm(Data.iterrows(), desc='Zipping', total=len(Data)):
    for d, region in tqdm(enumerate(row.RegionsFolder),
                          desc=f'Zipping regions of {os.path.dirname(row.LogFile[len(Root):])}',
                          total=len(row.RegionsFolder),
                          leave=False):
        zip_folder(region, row.LogFile, row['Scan date'])
        